# Examen blanc A : un encodeur pre-entraine aide-t-il quand on a peu de labels ?

**Tache** : classification de sentiment (phrase positive ou negative), jeu SST-2.

**Question de recherche** (celle du document BERT, section 5) : un encodeur Transformer pre-entraine
a-t-il besoin de moins d'exemples labellises que la meme architecture entrainee from scratch ?
Et que coute le gel de l'encodeur ?

**Modeles compares** (meme tokenisation, memes splits, meme budget) :
1. `scratch` : meme architecture que BERT, poids aleatoires, tout est entraine.
2. `frozen` : BERT pre-entraine, encodeur gele, seule la couche lineaire sur [CLS] est entrainee.
3. `finetune` : BERT pre-entraine, tout le reseau est entraine.
4. `gru` : GRU bidirectionnel entraine from scratch (reference recurrente, cf. TP4).

**Facteur varie** : nombre d'exemples d'entrainement labellises, n_train dans {100, 500, 2000}.

In [ ]:
# Sur Colab : Execution > Modifier le type d'execution > GPU (T4)
# !pip install -q datasets
import time, random
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

# Petit BERT officiel de Google (4 couches, dimension 256) : meme principe que bert-base,
# mais ~10 fois plus rapide, ce qui permet plusieurs runs en 50 minutes.
MODEL_NAME = "google/bert_uncased_L-4_H-256_A-4"
MAX_LEN = 64
QUICK = False  # True pour un test tres rapide (CPU)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 1. Donnees et splits

SST-2 n'a pas de labels publics sur son split `test`. On garde donc le split `validation`
officiel (872 phrases) comme **test**, regarde une seule fois a la fin, et on decoupe
notre propre **validation** dans le split `train` officiel.

In [ ]:
from datasets import load_dataset

raw = load_dataset("nyu-mll/glue", "sst2")
full_train = raw["train"].shuffle(seed=0)
val_raw = full_train.select(range(1000))            # validation : choix des epochs
pool_raw = full_train.select(range(1000, len(full_train)))  # reservoir d'exemples d'entrainement
test_raw = raw["validation"]                        # test : une seule fois, a la fin
print(len(val_raw), len(pool_raw), len(test_raw))
print(test_raw[0])
print("Proportion de positifs (test) :", np.mean(list(test_raw["label"])))

In [ ]:
from transformers import AutoTokenizer
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()  # masque le rapport de chargement des poids (tete MLM/NSP non utilisee)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
enc = tokenizer("A gripping, funny film.", padding="max_length", truncation=True, max_length=12)
print(tokenizer.convert_ids_to_tokens(enc["input_ids"]))  # [CLS] ... [SEP] [PAD] ...
print(enc["attention_mask"])                                # 1 = vrai token, 0 = padding


class SST2Dataset(Dataset):
    def __init__(self, hf_split):
        enc = tokenizer(list(hf_split["sentence"]), padding="max_length", truncation=True,
                        max_length=MAX_LEN, return_tensors="pt")
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.y = torch.tensor(list(hf_split["label"]), dtype=torch.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.attention_mask[idx], self.y[idx]


val_data = SST2Dataset(val_raw)
test_data = SST2Dataset(test_raw)
val_dataloader = DataLoader(val_data, batch_size=128)
test_dataloader = DataLoader(test_data, batch_size=128)

## 2. Modeles

Les deux classes suivent le design `MyTrainableModel` du TP1 / TP4 :
`forward`, `configure_optimizers`, `training_step`, `validation_step`.

In [ ]:
from transformers import AutoModel, AutoConfig


class BertClassifier(nn.Module):
    """Encodeur BERT + couche lineaire sur le vecteur de [CLS] (document BERT, section 4)."""

    def __init__(self, pretrained=True, freeze=False, n_classes=2, lr=None):
        super().__init__()
        if pretrained:
            self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        else:
            # Meme architecture, poids aleatoires : la seule difference est le pre-entrainement.
            self.encoder = AutoModel.from_config(AutoConfig.from_pretrained(MODEL_NAME))
        if freeze:
            for p in self.encoder.parameters():
                p.requires_grad = False
        self.linear = nn.Linear(self.encoder.config.hidden_size, n_classes)
        self.lr = lr if lr is not None else (1e-3 if freeze else 5e-5 if pretrained else 1e-4)
        self.loss_fn_mean = nn.CrossEntropyLoss(reduction='mean')
        self.loss_fn_sum = nn.CrossEntropyLoss(reduction='sum')

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_vector = out.last_hidden_state[:, 0]   # (batch, hidden) : position de [CLS]
        return self.linear(cls_vector)             # (batch, n_classes) : logits

    def configure_optimizers(self):
        params = [p for p in self.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=self.lr)

    def training_step(self, batch):
        ids, mask, y = batch
        return self.loss_fn_mean(self(ids, mask), y)

    def validation_step(self, batch):
        ids, mask, y = batch
        with torch.no_grad():
            pred = self(ids, mask)
            loss = self.loss_fn_sum(pred, y).item()
        return loss, pred.argmax(1), y


class GRUClassifier(nn.Module):
    """Embedding appris from scratch + GRU bidirectionnel (cf. TP4-2)."""

    def __init__(self, vocab_size, dim_emb=128, n_hidden=128, n_classes=2, lr=1e-3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim_emb, padding_idx=tokenizer.pad_token_id)
        self.gru = nn.GRU(dim_emb, n_hidden, batch_first=True, bidirectional=True)
        self.linear = nn.Linear(2 * n_hidden, n_classes)
        self.lr = lr
        self.loss_fn_mean = nn.CrossEntropyLoss(reduction='mean')
        self.loss_fn_sum = nn.CrossEntropyLoss(reduction='sum')

    def forward(self, input_ids, attention_mask):
        lens = attention_mask.sum(1).cpu()
        x = self.embedding(input_ids)                               # (batch, L, dim_emb)
        packed = nn.utils.rnn.pack_padded_sequence(x, lens, batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)                                   # h_n : (2, batch, n_hidden)
        h = torch.cat([h_n[0], h_n[1]], dim=1)                      # avant + arriere
        return self.linear(h)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

    training_step = BertClassifier.training_step
    validation_step = BertClassifier.validation_step

## 3. Entrainement

Le `Trainer` garde l'historique des pertes et, pour la selection, le modele de la meilleure
epoch en **validation** (early stopping simple). Le test n'intervient jamais ici.

In [ ]:
import copy


def evaluate(dataloader, model):
    model.eval()
    size = len(dataloader.dataset)
    loss, correct = 0., 0.
    for batch in dataloader:
        batch = [t.to(device) for t in batch]
        batch_loss, pred, y_true = model.validation_step(batch)
        loss += batch_loss
        correct += (pred == y_true).type(torch.float).sum().item()
    return loss / size, correct / size


class Trainer():
    def __init__(self, max_epochs):
        self.max_epochs = max_epochs

    def fit(self, model, train_dataloader, val_dataloader):
        model = model.to(device)
        optimizer = model.configure_optimizers()
        history = {"train_loss": [], "val_loss": [], "val_acc": []}
        best_acc, best_state = -1., None
        for epoch in range(self.max_epochs):
            model.train()
            running, n_batches = 0., 0
            for batch in train_dataloader:
                batch = [t.to(device) for t in batch]
                loss = model.training_step(batch)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                running += loss.item()
                n_batches += 1
            val_loss, val_acc = evaluate(val_dataloader, model)
            history["train_loss"].append(running / n_batches)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            if val_acc > best_acc:
                best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
        model.load_state_dict(best_state)
        return history


def make_model(name):
    if name == "scratch":
        return BertClassifier(pretrained=False)
    if name == "frozen":
        return BertClassifier(pretrained=True, freeze=True)
    if name == "finetune":
        return BertClassifier(pretrained=True, freeze=False)
    if name == "gru":
        return GRUClassifier(tokenizer.vocab_size)
    raise ValueError(name)


def run(name, n_train, seed, max_epochs):
    set_seed(seed)
    # Le sous-echantillon depend de la seed : la variabilite inclut le tirage des exemples.
    idx = np.random.RandomState(seed).choice(len(pool_raw), n_train, replace=False)
    train_data = SST2Dataset(pool_raw.select(idx))
    g = torch.Generator().manual_seed(seed)
    train_dataloader = DataLoader(train_data, batch_size=16, shuffle=True, generator=g)
    model = make_model(name)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    t0 = time.time()
    history = Trainer(max_epochs).fit(model, train_dataloader, val_dataloader)
    return model, history, n_trainable, time.time() - t0

## 4. Un premier run et ses courbes (verifier que tout marche avant de lancer la grille)

In [ ]:
model, history, n_trainable, secs = run("finetune", 100 if QUICK else 500, seed=0,
                                        max_epochs=2 if QUICK else 5)
print(f"{n_trainable:,} parametres entraines, {secs:.0f} s")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
a1.plot(history["train_loss"], label="train"); a1.plot(history["val_loss"], label="val")
a1.set_xlabel("epoch"); a1.set_ylabel("loss"); a1.legend()
a2.plot(history["val_acc"], color="green"); a2.set_xlabel("epoch"); a2.set_ylabel("val accuracy")
plt.show()

## 5. Grille d'experiences : 4 modeles x 3 tailles x 3 seeds

Budget : ~15 min sur un GPU T4 avec le petit BERT. Si le temps manque, reduire `SEEDS` a [0, 1]
ou retirer n_train=2000 : **mieux vaut moins de configurations, mais toutes repetees**.

In [ ]:
MODELS = ["scratch", "frozen", "finetune", "gru"]
SIZES = [40, 80] if QUICK else [100, 500, 2000]
SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = 1 if QUICK else 5

results = []  # (model, n_train, seed, val_acc_best, n_trainable, seconds)
for n_train in SIZES:
    for name in MODELS:
        for seed in SEEDS:
            model, history, n_trainable, secs = run(name, n_train, seed, EPOCHS)
            results.append((name, n_train, seed, max(history["val_acc"]), n_trainable, secs))
            print(f"{name:9s} n={n_train:5d} seed={seed}  val_acc={max(history['val_acc']):.3f}  ({secs:.0f} s)")

In [ ]:
import pandas as pd

df = pd.DataFrame(results, columns=["model", "n_train", "seed", "val_acc", "n_trainable", "seconds"])
summary = df.groupby(["model", "n_train"]).agg(mean=("val_acc", "mean"), std=("val_acc", "std"),
                                               seconds=("seconds", "mean")).reset_index()
print(summary.pivot(index="model", columns="n_train", values="mean").round(3))
print(summary.pivot(index="model", columns="n_train", values="std").round(3))

for name in MODELS:
    s = summary[summary.model == name]
    plt.errorbar(s.n_train, s["mean"], yerr=s["std"].fillna(0), marker="o", capsize=3, label=name)
plt.xscale("log"); plt.xlabel("n_train (echelle log)"); plt.ylabel("val accuracy (moyenne +/- ecart-type)")
plt.axhline(0.5, color="grey", ls=":", lw=1)
plt.legend(); plt.show()

## 6. Test : une seule fois, pour les modeles finaux

On reentraine chaque modele avec la plus grande taille (choix fige **avant** de regarder le test)
et on rapporte l'accuracy test. On ne revient plus ensuite sur aucun choix.

In [ ]:
n_final = SIZES[-1]
for name in MODELS:
    model, _, _, _ = run(name, n_final, seed=0, max_epochs=EPOCHS)
    test_loss, test_acc = evaluate(test_dataloader, model)
    print(f"TEST  {name:9s} n_train={n_final}: accuracy {100 * test_acc:.1f}%")